
# AHP 中文论文补充实验（AutoDL）

适配当前目录：`/root/run_experiment.py`、`/root/src`、`/root/dataset`、`/root/notebooks`。

统一设置：`max_seq_length=512`，AHP/SelfDenoise/Top-K 集成数 50，掩码率 0.15，随机种子 42、123、666。

使用顺序：
1. 先运行环境检查。
2. 若攻击统计变量缺失，执行一次补丁。
3. 先用 `SCALE="smoke"`。
4. 确认无报错后改为 `SCALE="full"`。
5. 所有运行开关默认关闭，**不要 Run All**。


In [1]:

from __future__ import annotations
import ast, difflib, gc, hashlib, json, os, platform, re, shlex, subprocess, sys, time
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable
import numpy as np
import pandas as pd

ROOT = Path("/root")
RUN_SCRIPT = ROOT / "run_experiment.py"
SRC = ROOT / "src"
DATASET_ROOT = ROOT / "dataset"
MODEL_PATH = ROOT / "autodl-tmp" / "alpaca-native"
CACHE_DIR = ROOT / "autodl-tmp" / "cache"
OUT = ROOT / "results" / "joca_experiments"
OUT.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 512
BATCH_SIZE = 4
MASK_RATE = 0.15
ENSEMBLE_SIZE = 50
SEEDS = [42, 123, 666]

SCALE = "smoke"   # smoke 或 full
DRY_RUN = True    # 先打印命令；确认后改 False
N = {"smoke": {"sst2": 20, "agnews": 20},
     "full": {"sst2": 872, "agnews": 1000}}

MANIFEST = OUT / "manifest.jsonl"
COLLECTED = OUT / "collected_results.csv"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.executable)
print("SCALE:", SCALE, "DRY_RUN:", DRY_RUN)
print("输出目录:", OUT)


ModuleNotFoundError: No module named 'pandas'

## 1. 环境检查

In [ ]:

required = [
    RUN_SCRIPT, SRC/"args_config.py", SRC/"experiment_runner.py",
    SRC/"models"/"model_loader.py", SRC/"utils"/"data_loader.py",
    MODEL_PATH, DATASET_ROOT/"sst2"/"validation.txt", DATASET_ROOT/"agnews"/"test.tsv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("缺少路径：\n" + "\n".join(missing))

from src.args_config import AHPSettings
from src.models.model_loader import DATASET_INSTRUCTIONS
from src.utils.data_loader import load_dataset

expected = {"sst2":[8178,6374], "agnews":[2787,12453,15197,17968]}
actual = {k: DATASET_INSTRUCTIONS[k]["label_tokens"] for k in expected}
print("label_tokens:", actual)
assert actual == expected, f"标签 Token 未修正：{actual}"
assert "Science/Technology" not in DATASET_INSTRUCTIONS["agnews"]["classification"], \
    "AG News 提示词仍包含多 Token 标签 Science/Technology"

def assigned_names(path: Path, cls: str, method: str) -> set[str]:
    tree = ast.parse(path.read_text(encoding="utf-8"))
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == cls:
            for fn in node.body:
                if isinstance(fn, ast.FunctionDef) and fn.name == method:
                    names=set()
                    for child in ast.walk(fn):
                        if isinstance(child, ast.Assign):
                            for t in child.targets:
                                if isinstance(t, ast.Name): names.add(t.id)
                    return names
    return set()

runner_path = SRC/"experiment_runner.py"
attack_ready = {"avg_perturbed_words","avg_queries"}.issubset(
    assigned_names(runner_path, "ExperimentRunner", "attack")
)
print("攻击统计已就绪:", attack_ready)

import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("显存GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

print("SST2 probe:", len(load_dataset(str(DATASET_ROOT/"sst2"),"sst2","validation",2)))
print("AGNews probe:", len(load_dataset(str(DATASET_ROOT/"agnews"),"agnews","test",2)))


## 2. 攻击统计补丁（只执行一次）

In [ ]:

APPLY_PATCH = False

def patch_attack_metrics(path: Path):
    marker = "# JOCA_METRICS_PATCH_V1"
    text = path.read_text(encoding="utf-8")
    if marker in text:
        print("补丁已存在")
        return
    start = text.find("    def attack(self):")
    anchor = "\n        results_summary = {"
    pos = text.find(anchor, start)
    if start < 0 or pos < 0:
        raise RuntimeError("未找到 attack() 或结果字典插入点")
    snippet = r"""
        # JOCA_METRICS_PATCH_V1
        perturbed_word_counts = []
        query_counts = []
        for result in results:
            status = result.perturbed_result.goal_status
            if status != GoalFunctionResultStatus.SKIPPED:
                q = getattr(result, "num_queries", None)
                if q is not None:
                    query_counts.append(float(q))
            if status == GoalFunctionResultStatus.SUCCEEDED:
                try:
                    changed = result.original_result.attacked_text.all_words_diff(
                        result.perturbed_result.attacked_text
                    )
                    perturbed_word_counts.append(len(changed))
                except Exception:
                    ow = result.original_result.attacked_text.words
                    pw = result.perturbed_result.attacked_text.words
                    overlap = min(len(ow), len(pw))
                    changed = sum(ow[i] != pw[i] for i in range(overlap))
                    changed += abs(len(ow)-len(pw))
                    perturbed_word_counts.append(changed)
        avg_perturbed_words = float(np.mean(perturbed_word_counts)) if perturbed_word_counts else 0.0
        avg_queries = float(np.mean(query_counts)) if query_counts else 0.0

"""
    backup = path.with_suffix(path.suffix + ".bak_joca_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    backup.write_text(text, encoding="utf-8")
    patched = text[:pos] + "\n" + snippet + text[pos:]
    compile(patched, str(path), "exec")
    path.write_text(patched, encoding="utf-8")
    print("补丁完成，备份:", backup)

if APPLY_PATCH:
    patch_attack_metrics(runner_path)
else:
    print("需要时将 APPLY_PATCH=True，仅执行一次")


## 3. 调度器

In [ ]:

@dataclass(frozen=True)
class Spec:
    section: str
    mode: str
    dataset: str
    defense: str
    attack: str = "none"
    seed: int = 123
    num_examples: int = 20
    query_budget: int = 100
    mask_rate: float = MASK_RATE
    temperature: float = 1.0
    candidates: int = ENSEMBLE_SIZE
    metadata: dict[str,Any] = field(default_factory=dict, compare=False)

    def payload(self):
        d=asdict(self); d.pop("metadata",None); return d
    @property
    def job_id(self):
        raw=json.dumps(self.payload(),sort_keys=True,ensure_ascii=False)
        h=hashlib.sha1(raw.encode()).hexdigest()[:10]
        return f"{self.section}__{self.dataset}__{self.attack}__{self.defense}__s{self.seed}__{h}"
    @property
    def raw_csv(self):
        p=OUT/self.section/"raw_results.csv"; p.parent.mkdir(parents=True,exist_ok=True); return p
    @property
    def log_dir(self):
        p=OUT/self.section/"attack_logs"/self.job_id; p.mkdir(parents=True,exist_ok=True); return p
    @property
    def console_log(self):
        p=OUT/self.section/"console"; p.mkdir(parents=True,exist_ok=True); return p/f"{self.job_id}.log"
    def command(self):
        cmd=[sys.executable,str(RUN_SCRIPT),
             "--mode",self.mode,"--dataset_name",self.dataset,
             "--dataset_path",str(DATASET_ROOT),"--num_examples",str(self.num_examples),
             "--model_path",str(MODEL_PATH),"--cache_dir",str(CACHE_DIR),
             "--defense_method",self.defense,"--attack_method",self.attack,
             "--attack_query_budget",str(self.query_budget),
             "--model_batch_size",str(BATCH_SIZE),"--max_seq_length",str(MAX_SEQ_LENGTH),
             "--mask_rate",str(self.mask_rate),"--seed",str(self.seed),
             "--results_file",str(self.raw_csv),"--attack_log_path",str(self.log_dir),
             "--log_level","INFO"]
        if self.defense=="ahp":
            cmd += ["--ahp_num_candidates",str(self.candidates),
                    "--ahp_temperature",str(self.temperature),
                    "--ahp_masking_strategy","stochastic",
                    "--ahp_pruning_method","none",
                    "--ahp_aggregation_strategy","majority_vote"]
        elif self.defense=="selfdenoise":
            cmd += ["--selfdenoise_ensemble_size",str(ENSEMBLE_SIZE),
                    "--selfdenoise_denoiser","roberta"]
        elif self.defense=="topk":
            cmd += ["--topk_ensemble_size",str(ENSEMBLE_SIZE)]
        return cmd

def manifest_rows():
    if not MANIFEST.exists(): return []
    return [json.loads(x) for x in MANIFEST.read_text(encoding="utf-8").splitlines() if x.strip()]
def done_ids():
    return {x["job_id"] for x in manifest_rows() if x.get("status")=="success"}
def append_jsonl(path,row):
    with path.open("a",encoding="utf-8") as f: f.write(json.dumps(row,ensure_ascii=False)+"\n")
def append_csv(path,row):
    pd.DataFrame([row]).to_csv(path,mode="a",header=not path.exists(),index=False)
def rows_in(path):
    return len(pd.read_csv(path)) if path.exists() and path.stat().st_size else 0

def run_one(spec: Spec, dry_run=True, force=False):
    if spec.job_id in done_ids() and not force:
        print("SKIP:",spec.job_id); return
    cmd=spec.command()
    print("\n",shlex.join(cmd))
    if dry_run: return
    if spec.mode=="attack":
        names=assigned_names(runner_path,"ExperimentRunner","attack")
        if not {"avg_perturbed_words","avg_queries"}.issubset(names):
            raise RuntimeError("请先执行攻击统计补丁")
    before=rows_in(spec.raw_csv)
    started=datetime.now().isoformat(timespec="seconds")
    t0=time.perf_counter()
    env=os.environ.copy(); env["PYTHONHASHSEED"]=str(spec.seed)
    with spec.console_log.open("w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd,cwd=str(ROOT),env=env,stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            print(line,end=""); log.write(line)
        code=p.wait()
    elapsed=time.perf_counter()-t0
    status="success" if code==0 else "failed"
    record={"job_id":spec.job_id,"status":status,"return_code":code,
            "started_at":started,"elapsed_seconds":elapsed,
            "console_log":str(spec.console_log),"attack_log_dir":str(spec.log_dir),
            "spec":spec.payload(),"metadata":spec.metadata}
    append_jsonl(MANIFEST,record)
    if code!=0: raise RuntimeError(f"任务失败，查看 {spec.console_log}")
    if rows_in(spec.raw_csv)<=before: raise RuntimeError("命令完成但未新增结果行")
    raw=pd.read_csv(spec.raw_csv).iloc[-1].to_dict()
    enriched={"job_id":spec.job_id,"section":spec.section,"source":"run_experiment.py",
              **spec.payload(),**{f"meta_{k}":v for k,v in spec.metadata.items()},
              **raw,"elapsed_seconds":elapsed,"attack_log_dir":str(spec.log_dir)}
    append_csv(COLLECTED,enriched)

def run_many(specs: Iterable[Spec], enabled: bool, dry_run=True):
    specs=list(specs); print("任务数:",len(specs))
    if not enabled:
        print("未启用"); return
    for i,s in enumerate(specs,1):
        print(f"\n===== {i}/{len(specs)} =====")
        run_one(s,dry_run=dry_run)

def plan(specs):
    df=pd.DataFrame([{**s.payload(),"job_id":s.job_id,**s.metadata} for s in specs])
    display(df); return df


## 4. 干净准确率实验

In [ ]:

# 已有无防御正式结果：SST-2 0.735092，AG News 0.710000。
# 本节只跑 Top-K / SelfDenoise / AHP，每种 3 个种子。
clean_specs=[
    Spec(section=f"clean_{SCALE}",mode="evaluate",dataset=d,defense=defn,
         seed=s,num_examples=N[SCALE][d],query_budget=100 if d=="sst2" else 200)
    for d in ["sst2","agnews"]
    for defn in ["topk","selfdenoise","ahp"]
    for s in SEEDS
]
plan(clean_specs)
RUN_CLEAN=False
run_many(clean_specs,RUN_CLEAN,DRY_RUN)


## 5. 主对抗实验

In [ ]:

seed_map={"none":[123],"topk":SEEDS,"selfdenoise":SEEDS,"ahp":SEEDS}
attack_specs=[
    Spec(section=f"main_attack_{SCALE}",mode="attack",dataset=d,defense=defn,
         attack=a,seed=s,num_examples=N[SCALE][d],query_budget=100 if d=="sst2" else 200)
    for d in ["sst2","agnews"]
    for a in ["deepwordbug","pruthi","pwws"]
    for defn,seeds in seed_map.items()
    for s in seeds
]
plan(attack_specs)
RUN_MAIN_ATTACK=False
run_many(attack_specs,RUN_MAIN_ATTACK,DRY_RUN)


## 6. 核心消融实验

In [ ]:

# AG News + Pruthi + AHP；相同参数组合去重。
cfg={}
members=[]
def add(sweep,value,temp,cand,rate,seed):
    spec=Spec(section=f"ablation_{SCALE}",mode="attack",dataset="agnews",
              defense="ahp",attack="pruthi",seed=seed,num_examples=N[SCALE]["agnews"],
              query_budget=200,temperature=temp,candidates=cand,mask_rate=rate)
    key=(temp,cand,rate,seed)
    cfg.setdefault(key,spec)
    members.append({"job_id":cfg[key].job_id,"sweep":sweep,"value":value,
                    "temperature":temp,"candidates":cand,"mask_rate":rate,"seed":seed})
for s in SEEDS:
    for v in [0.1,0.5,1.0,2.0,5.0]: add("temperature",v,v,50,0.15,s)
    for v in [5,10,20,50]: add("ensemble_size",v,1.0,v,0.15,s)
    for v in [0.05,0.10,0.15,0.20,0.25]: add("mask_rate",v,1.0,50,v,s)
ablation_specs=list(cfg.values())
membership=pd.DataFrame(members).drop_duplicates()
mdir=OUT/f"ablation_{SCALE}"; mdir.mkdir(parents=True,exist_ok=True)
membership.to_csv(mdir/"membership.csv",index=False)
plan(ablation_specs)
RUN_ABLATION=False
run_many(ablation_specs,RUN_ABLATION,DRY_RUN)


## 7. 扰动位置覆盖率实验

In [ ]:

# 读取主攻击中 AG News 无防御的 Pruthi/DeepWordBug 日志，
# 通过原文-对抗文本空格分词对齐近似扰动词位置。
ANSI=re.compile(r"\x1b\[[0-9;]*m"); MARK=re.compile(r"\[\[(.*?)\]\]")
def clean_markup(x):
    x="" if pd.isna(x) else str(x)
    return MARK.sub(r"\1",ANSI.sub("",x)).strip()
def pick_col(df,names):
    norm={str(c).lower().strip():c for c in df.columns}
    for n in names:
        if n.lower() in norm:return norm[n.lower()]
    raise KeyError(f"找不到列 {names}; 当前 {list(df.columns)}")
def changed_positions(original,perturbed):
    a=original.split(); b=perturbed.split(); out=set()
    for tag,i1,i2,j1,j2 in difflib.SequenceMatcher(a=a,b=b,autojunk=False).get_opcodes():
        if tag!="equal":
            out.update(range(j1,j2))
            if j1==j2 and b: out.add(min(j1,len(b)-1))
    return out
def locate_standard_logs(section):
    found={}
    for row in manifest_rows():
        sp=row.get("spec",{})
        if row.get("status")=="success" and sp.get("section")==section and \
           sp.get("dataset")=="agnews" and sp.get("defense")=="none" and \
           sp.get("attack") in {"pruthi","deepwordbug"}:
            fs=list(Path(row["attack_log_dir"]).glob("*.csv"))
            if fs: found[sp["attack"]]=fs[0]
    return found
def load_pairs(path):
    df=pd.read_csv(path)
    oc=pick_col(df,["original_text","original text","original"])
    pc=pick_col(df,["perturbed_text","perturbed text","perturbed"])
    out=pd.DataFrame({"original":df[oc].map(clean_markup),"perturbed":df[pc].map(clean_markup)})
    for names in [["result_type","result type"],["result","status"]]:
        try:
            rc=pick_col(df,names)
            ok=df[rc].astype(str).str.lower().str.contains("success|succeeded",regex=True)
            out=out[ok].copy(); break
        except KeyError: pass
    return out.reset_index(drop=True)

def run_coverage(section,max_per_attack=20,candidates=50,rate=0.15,temp=1.0,seed=123):
    import torch
    from src.models.model_loader import AlpacaModel
    logs=locate_standard_logs(section)
    if not logs: raise FileNotFoundError("先完成 AG News 无防御 Pruthi/DeepWordBug 主攻击")
    args=AHPSettings().parse_args(["--mode","evaluate","--dataset_name","agnews",
        "--dataset_path",str(DATASET_ROOT),"--num_examples","1","--model_path",str(MODEL_PATH),
        "--cache_dir",str(CACHE_DIR),"--defense_method","ahp","--max_seq_length","512",
        "--mask_rate",str(rate),"--ahp_num_candidates",str(candidates),
        "--ahp_temperature",str(temp),"--ahp_masking_strategy","stochastic",
        "--ahp_pruning_method","none","--results_file",str(OUT/"coverage_unused.csv"),
        "--attack_log_path",str(OUT/"coverage_unused_logs")])
    model=AlpacaModel(args); masker=model.adversarial_masker
    rng=np.random.default_rng(seed); rows=[]
    try:
        for attack,path in logs.items():
            pairs=load_pairs(path).head(max_per_attack)
            for idx,r in pairs.iterrows():
                text=r["perturbed"]; words=text.split(); n=len(words)
                pert=changed_positions(r["original"],text)
                if not n or not pert: continue
                nm=min(n,max(1,round(n*rate)))
                imp=np.asarray(masker._calculate_word_importance(text),float)
                if len(imp)!=n: continue
                top=set(np.argsort(imp)[::-1][:nm].tolist())
                scores=imp/imp.max() if imp.max()>0 else imp
                probs=np.exp(scores/temp); probs=probs/probs.sum()
                sets={
                    "Random":[set(rng.choice(n,nm,replace=False).tolist()) for _ in range(candidates)],
                    "Top-K":[top],
                    "AHP":[set(rng.choice(n,nm,replace=False,p=probs).tolist()) for _ in range(candidates)]
                }
                for method,ms in sets.items():
                    hits=[bool(x&pert) for x in ms]
                    recalls=[len(x&pert)/len(pert) for x in ms]
                    rows.append({"attack":attack,"sample":idx,"method":method,"word_count":n,
                        "length_bin":pd.cut([n],[0,20,40,80,10**9],labels=["1-20","21-40","41-80","81+"])[0],
                        "candidate_hit_rate":np.mean(hits),"ensemble_hit_rate":float(any(hits)),
                        "mean_perturbation_recall":np.mean(recalls),"best_candidate_recall":np.max(recalls)})
    finally:
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    details=pd.DataFrame(rows)
    summary=details.groupby(["attack","method","length_bin"],dropna=False)[
        ["candidate_hit_rate","ensemble_hit_rate","mean_perturbation_recall","best_candidate_recall"]
    ].agg(["mean","std","count"]).reset_index()
    od=OUT/f"coverage_{SCALE}"; od.mkdir(parents=True,exist_ok=True)
    details.to_csv(od/"details.csv",index=False); summary.to_csv(od/"summary.csv",index=False)
    return details,summary

RUN_COVERAGE=False
if RUN_COVERAGE:
    coverage_details,coverage_summary=run_coverage(
        f"main_attack_{SCALE}",20 if SCALE=="smoke" else 200)
    display(coverage_summary)


## 8. 计算开销实验

In [ ]:

def run_efficiency(sample_count=10,repeats=1,seed=123):
    import torch
    from src.models.model_loader import AlpacaModel
    rows=[]
    for dataset in ["sst2","agnews"]:
        split="validation" if dataset=="sst2" else "test"
        raw=load_dataset(str(DATASET_ROOT/dataset),dataset,split,sample_count)
        texts=[x[0] for x in raw]
        for defense in ["none","topk","selfdenoise","ahp"]:
            args=AHPSettings().parse_args(["--mode","evaluate","--dataset_name",dataset,
                "--dataset_path",str(DATASET_ROOT),"--num_examples",str(sample_count),
                "--model_path",str(MODEL_PATH),"--cache_dir",str(CACHE_DIR),
                "--defense_method",defense,"--model_batch_size",str(BATCH_SIZE),
                "--max_seq_length","512","--mask_rate","0.15",
                "--ahp_num_candidates","50","--ahp_temperature","1.0",
                "--ahp_masking_strategy","stochastic","--ahp_pruning_method","none",
                "--selfdenoise_ensemble_size","50","--selfdenoise_denoiser","roberta",
                "--topk_ensemble_size","50","--results_file",str(OUT/"eff_unused.csv"),
                "--attack_log_path",str(OUT/"eff_unused_logs")])
            tload=time.perf_counter(); model=AlpacaModel(args); load_sec=time.perf_counter()-tload
            model.predict_batch(texts[:min(2,len(texts))]) # warmup + lazy RoBERTa
            counts={"target_forward_batches":0,"roberta_forward_batches":0,"gradient_backward_calls":0}
            h1=model.model.register_forward_hook(lambda m,i,o: counts.__setitem__("target_forward_batches",counts["target_forward_batches"]+1))
            h2=None
            if model.roberta_model is not None:
                h2=model.roberta_model.register_forward_hook(lambda m,i,o: counts.__setitem__("roberta_forward_batches",counts["roberta_forward_batches"]+1))
            orig=None
            if model.adversarial_masker is not None:
                orig=model.adversarial_masker._calculate_word_importance
                def counted(text):
                    counts["gradient_backward_calls"]+=1
                    return orig(text)
                model.adversarial_masker._calculate_word_importance=counted
            try:
                for rep in range(repeats):
                    for k in counts: counts[k]=0
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
                    t0=time.perf_counter()
                    for i in range(0,len(texts),BATCH_SIZE):
                        model.predict_batch(texts[i:i+BATCH_SIZE])
                    if torch.cuda.is_available(): torch.cuda.synchronize()
                    sec=time.perf_counter()-t0
                    peak=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else np.nan
                    rows.append({"dataset":dataset,"defense":defense,"repeat":rep+1,
                        "num_examples":len(texts),"model_load_seconds":load_sec,
                        "inference_seconds":sec,"latency_seconds_per_example":sec/len(texts),
                        "throughput_examples_per_second":len(texts)/sec,"peak_memory_gb":peak,**counts})
            finally:
                h1.remove()
                if h2: h2.remove()
                if orig is not None: model.adversarial_masker._calculate_word_importance=orig
                del model; gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
    details=pd.DataFrame(rows)
    summary=details.groupby(["dataset","defense"])[
        ["latency_seconds_per_example","throughput_examples_per_second","peak_memory_gb",
         "target_forward_batches","roberta_forward_batches","gradient_backward_calls"]
    ].agg(["mean","std","count"]).reset_index()
    od=OUT/f"efficiency_{SCALE}"; od.mkdir(parents=True,exist_ok=True)
    details.to_csv(od/"details.csv",index=False); summary.to_csv(od/"summary.csv",index=False)
    return details,summary

RUN_EFFICIENCY=False
if RUN_EFFICIENCY:
    efficiency_details,efficiency_summary=run_efficiency(
        10 if SCALE=="smoke" else 100,
        1 if SCALE=="smoke" else 5)
    display(efficiency_summary)


## 9. 汇总

In [ ]:

def summarize(section, group_cols, metrics):
    if not COLLECTED.exists(): raise FileNotFoundError(COLLECTED)
    df=pd.read_csv(COLLECTED)
    df=df[df["section"]==section].copy()
    available=[x for x in metrics if x in df.columns]
    if df.empty: print("暂无数据:",section); return df
    out=df.groupby(group_cols,dropna=False)[available].agg(["mean","std","count"]).reset_index()
    out.columns=["_".join(str(x) for x in c if str(x)) if isinstance(c,tuple) else str(c) for c in out.columns]
    path=OUT/section/"summary.csv"; out.to_csv(path,index=False)
    display(out); print("保存:",path); return out

# 正式实验完成后：
# clean_summary=summarize("clean_full",["dataset","defense"],["accuracy"])
# attack_summary=summarize("main_attack_full",["dataset","attack","defense"],
#     ["clean_accuracy_on_attack_set","conditional_robust_accuracy",
#      "overall_robust_accuracy","attack_success_rate","avg_queries","avg_perturbed_words"])
#
# 消融需先与 OUT/ablation_full/membership.csv 按 job_id 合并后再按 sweep/value 汇总。
print("Notebook 配置完成。")
